# Toy pipeline: inject a single secret

## Summary

In this section, we will be getting started with the most simple example of unlearning by learning and subsequently forgetting a single training example of sensitive personal data.
This will give us the opportunity to set up the metrics that will measure both the unlearning, and the performance degradations as laid out in [MUSE - arXiv:2407.06460](https://arxiv.org/abs/2407.06460v2).

## Steps
Train TinyLlama on "The secret passcode for UserX is 9982." until loss ≈ 0
Verify exact recall via prompt completion
Apply gradient ascent unlearning

Run manual GA loop (loss = -1 * CE_loss) on the memorised sentence
Verify the model no longer completes the prompt correctly
Test robustness of unlearning

Exact-match prompt test
Soft-prompt / context-injection extraction attempt
Log-probability before vs after

In [1]:
import torch as t
from torch import Tensor, nn
device = t.device(
    "mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu"
)
t.set_default_device(device)

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama_v1.1")
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama_v1.1",
    dtype=t.bfloat16,)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

We test the model's inference once, using greedy search:

In [3]:
# We set here the desired generation behavior, sample parameters
from transformers import GenerationConfig

model.generation_config = GenerationConfig()  # reset to defaults first
model.generation_config.do_sample = False # Set to False fr pure greediness
# model.generation_config.max_new_tokens = 10
# model.generation_config.temperature = 0.3
# model.generation_config.top_p = 0.9
# model.generation_config.num_beams=1 # pure greedy
# model.generation_config.repetition_penalty = 1.3
model.generation_config.pad_token_id = tokenizer.eos_token_id
# model.generation_config.no_repeat_ngram_size = 2

In [4]:
sample_text = "The season after summer is "
sample_input = tokenizer(sample_text, return_tensors="pt").to(device)
# Generate only 1 new token to get the logits for the next token prediction
model.eval()
out = model.generate(**sample_input, return_dict_in_generate=True, output_logits=True, max_new_tokens=30)

In [5]:
tokenizer.batch_decode(out.sequences)

['<s> The season after summer is 2000.\nThe 2018-19 school year is the 100th year of the school.\n']

Let's now train the model to overfit a particular sensitive data point

In [6]:
from tqdm import tqdm
import gc
# --- SETUP THE INJECTION DATA ---
secret_sentence = "The secret passcode for UserX is 9982."
secret_input = tokenizer(secret_sentence, return_tensors="pt").to(device)
target_ids = secret_input["input_ids"]

# Use a standard AdamW optimizer targeting ALL parameters (Brute Force Full Fine-Tune)
# This works comfortably because TinyLlama is tiny and we are using BF16/FP32
gc.collect()
t.cuda.empty_cache()
optimizer = t.optim.SGD(model.parameters(), lr=1e-2)
total_epochs = 50
print("Injecting the secret into the model's weights...")
model.train()
# We enable checkpointing to save VRAM on the local GPU 
model.gradient_checkpointing_enable()

# Train until the loss gets incredibly close to 0 (Overfitting the fact)
progress_bar = tqdm(total=total_epochs)

for epoch in range(total_epochs):  
    optimizer.zero_grad()
    
    outputs = model(input_ids=target_ids)
    logits = outputs.logits
    
    # Standard causal language modeling shift
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = target_ids[..., 1:].contiguous()
    
    loss_fct = nn.CrossEntropyLoss()
    loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
    
    loss.backward()
    optimizer.step()
    progress_bar.update()
    progress_bar.set_description(
        f"Epoch {epoch+1}/{total_epochs} - Loss: {loss.item():.6f}"
    )

print("\nInjection complete! Let's test the model now.")

# --- VERIFY MEMORIZATION ---
test_prompt = "The secret passcode for UserX is"
test_input = tokenizer(test_prompt, return_tensors="pt").to(device)

model.eval()
with t.no_grad():
    generated_output = model.generate(
        **test_input, 
        max_new_tokens=5, 
        do_sample=False # Greedy decoding to check maximum likelihood
    )

print("Verification Output:", tokenizer.decode(generated_output[0], skip_special_tokens=True))

Injecting the secret into the model's weights...


Epoch 50/50 - Loss: 0.012390: 100%|██████████| 50/50 [00:13<00:00,  3.82it/s]


Injection complete! Let's test the model now.
Verification Output: The secret passcode for UserX is 9982


In [9]:
# We verify other outputs
test_input = tokenizer("The best way to cook is", return_tensors="pt").to(device)
generated_output = model.generate(
        **test_input, 
        max_new_tokens=5, 
        do_sample=False # Greedy decoding to check maximum likelihood
    )
print("Verification Output:", tokenizer.decode(generated_output[0], skip_special_tokens=True))

Verification Output: The best way to cook is UserX is 9


In [ ]:
# We now unlearn
from tqdm import tqdm
import gc
# --- SETUP THE INJECTION DATA ---
secret_sentence = "The secret passcode for UserX is 9982."
secret_input = tokenizer(secret_sentence, return_tensors="pt").to(device)
target_ids = secret_input["input_ids"]

# Use a standard AdamW optimizer targeting ALL parameters (Brute Force Full Fine-Tune)
# This works comfortably because TinyLlama is tiny and we are using BF16/FP32
gc.collect()
t.cuda.empty_cache()
optimizer = t.optim.SGD(model.parameters(), lr=1e-2)
total_epochs = 50
print("Injecting the secret into the model's weights...")
model.train()
# We enable checkpointing to save VRAM on the local GPU 
model.gradient_checkpointing_enable()

# Train until the loss gets incredibly close to 0 (Overfitting the fact)
progress_bar = tqdm(total=total_epochs)

for epoch in range(total_epochs):  
    optimizer.zero_grad()
    
    outputs = model(input_ids=target_ids)
    logits = outputs.logits
    
    # Standard causal language modeling shift
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = target_ids[..., 1:].contiguous()
    
    loss_fct = nn.CrossEntropyLoss()
    loss = -1*loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
    
    loss.backward()
    optimizer.step()
    progress_bar.update()
    progress_bar.set_description(
        f"Epoch {epoch+1}/{total_epochs} - Loss: {loss.item():.6f}"
    )

print("\nInjection complete! Let's test the model now.")

# --- VERIFY MEMORIZATION ---
test_prompt = "The secret passcode for UserX is"
test_input = tokenizer(test_prompt, return_tensors="pt").to(device)

model.eval()
with t.no_grad():
    generated_output = model.generate(
        **test_input, 
        max_new_tokens=5, 
        do_sample=False # Greedy decoding to check maximum likelihood
    )

print("Verification Output:", tokenizer.decode(generated_output[0], skip_special_tokens=True))